In [1]:
"""
Embedding-Based Community Detection Testing
For Master's Research Project
Author: MOUDDEN Hamza

Methods: DeepWalk, Node2Vec, GraphSAGE (Node Embeddings + k-means)
Metrics: Modularity, NMI, ARI
"""

import os
import time
import numpy as np
import pandas as pd
import networkx as nx
from collections import defaultdict
from typing import Dict
import ast
import warnings
warnings.filterwarnings('ignore')

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score

# ============================================================
# 1. PATH HANDLING AND DATA LOADING
# ============================================================

def get_data_path(filename: str) -> str:
    """Get full path to a dataset in the data folder"""
    possible_paths = [
        os.path.join("data", filename),
        os.path.join("../data", filename),
        os.path.join(os.getcwd(), "data", filename),
        os.path.join(os.path.dirname(os.getcwd()), "data", filename),
        filename
    ]
    
    for path in possible_paths:
        if os.path.exists(path):
            return path
    
    return os.path.join("data", filename)

def load_ground_truth_from_txt(gt_filename: str) -> Dict[int, int]:
    """Load ground truth communities from a text file"""
    ground_truth = {}
    
    if not gt_filename.endswith('.txt'):
        gt_filename = gt_filename + '.txt'
    
    path = get_data_path(gt_filename)
    
    if not os.path.exists(path):
        return ground_truth
    
    try:
        with open(path, 'r') as f:
            content = f.read().strip()
        
        if content.startswith('[') and content.endswith(']'):
            try:
                labels_list = ast.literal_eval(content)
                unique_labels = {}
                next_id = 0
                for node_idx, label in enumerate(labels_list):
                    if label not in unique_labels:
                        unique_labels[label] = next_id
                        next_id += 1
                    ground_truth[node_idx] = unique_labels[label]
            except:
                content = content[1:-1]
                parts = [p.strip().strip('"').strip("'") for p in content.split(',')]
                unique_labels = {}
                next_id = 0
                for node_idx, label in enumerate(parts):
                    if label not in unique_labels:
                        unique_labels[label] = next_id
                        next_id += 1
                    ground_truth[node_idx] = unique_labels[label]
        else:
            lines = content.split('\n')
            for line in lines:
                line = line.strip()
                if line and not line.startswith('#'):
                    parts = line.split()
                    if len(parts) >= 2:
                        try:
                            node = int(parts[0])
                            community = int(parts[1])
                            ground_truth[node] = community
                        except:
                            pass
    
    except Exception as e:
        print(f"  Warning: Could not load ground truth: {e}")
    
    return ground_truth

def load_graph_safe(filepath: str) -> nx.Graph:
    """Safely load a GML file and return an undirected graph"""
    try:
        G = nx.read_gml(filepath)
    except:
        try:
            G = nx.read_gml(filepath, label='id')
        except:
            try:
                G = nx.read_edgelist(filepath)
            except Exception as e:
                raise Exception(f"Cannot load graph: {e}")
    
    if nx.is_directed(G):
        G = G.to_undirected()
    
    G.remove_edges_from(nx.selfloop_edges(G))
    
    node_mapping = {node: i for i, node in enumerate(sorted(G.nodes()))}
    G = nx.relabel_nodes(G, node_mapping)
    
    return G

# ============================================================
# 2. DATASET LOADING FUNCTIONS
# ============================================================

def load_karate_club():
    path = get_data_path("karate.gml")
    G = load_graph_safe(path)
    gt = load_ground_truth_from_txt("karate_GR")
    if not gt:
        for node in G.nodes():
            gt[node] = 0 if node != 33 else 1
    return G, gt

def load_dolphins():
    path = get_data_path("Dolphins.gml")
    G = load_graph_safe(path)
    gt = load_ground_truth_from_txt("dolphins_GR")
    return G, gt

def load_football():
    path = get_data_path("Football.gml")
    G = load_graph_safe(path)
    gt = load_ground_truth_from_txt("football_GR")
    return G, gt

def load_polbooks():
    path = get_data_path("Polbooks.gml")
    G = load_graph_safe(path)
    gt = load_ground_truth_from_txt("polbooks_GT")
    return G, gt

def load_primary_school():
    path = get_data_path("PS.gml")
    if not os.path.exists(path):
        path = get_data_path("Thiers.gml")
    
    G = load_graph_safe(path)
    
    # Try to load from classname in GML first
    ground_truth = {}
    for node in G.nodes():
        if 'classname' in G.nodes[node]:
            classname = G.nodes[node]['classname']
            # Map classname to integer
            class_map = {'1A':0, '1B':1, '2A':2, '2B':3, '3A':4, '3B':5, '4A':6, '4B':7, '5A':8, '5B':9}
            if classname in class_map:
                ground_truth[node] = class_map[classname]
    
    if not ground_truth:
        ground_truth = load_ground_truth_from_txt("Thiers_GR")
    if not ground_truth:
        ground_truth = load_ground_truth_from_txt("PSD1_GR")
    if not ground_truth:
        ground_truth = load_ground_truth_from_txt("PSD2_GR")
    
    if not ground_truth:
        for node in G.nodes():
            ground_truth[node] = node % 10
    
    return G, ground_truth

def load_vs13():
    path = get_data_path("VS13.gml")
    G = load_graph_safe(path)
    gt = load_ground_truth_from_txt("VS13_GR")
    return G, gt

def load_vs15():
    path = get_data_path("VS15.gml")
    G = load_graph_safe(path)
    gt = load_ground_truth_from_txt("VS15_GR")
    return G, gt

# ============================================================
# 3. EVALUATION METRICS
# ============================================================

def calculate_modularity(G: nx.Graph, partition: Dict[int, int]) -> float:
    """Calculate modularity score"""
    m = G.number_of_edges()
    if m == 0:
        return 0.0
    
    communities = defaultdict(set)
    for node, comm in partition.items():
        communities[comm].add(node)
    
    modularity = 0.0
    for comm_nodes in communities.values():
        subgraph = G.subgraph(comm_nodes)
        l_c = subgraph.number_of_edges()
        d_c = sum(G.degree(node) for node in comm_nodes)
        modularity += (l_c / m) - (d_c / (2 * m)) ** 2
    
    return modularity

def calculate_nmi(partition1: Dict[int, int], partition2: Dict[int, int]) -> float:
    nodes = sorted(set(partition1.keys()) & set(partition2.keys()))
    if not nodes:
        return 0.0
    
    labels1 = [partition1[node] for node in nodes]
    labels2 = [partition2[node] for node in nodes]
    
    return normalized_mutual_info_score(labels1, labels2)

def calculate_ari(partition1: Dict[int, int], partition2: Dict[int, int]) -> float:
    nodes = sorted(set(partition1.keys()) & set(partition2.keys()))
    if not nodes:
        return 0.0
    
    labels1 = [partition1[node] for node in nodes]
    labels2 = [partition2[node] for node in nodes]
    
    return adjusted_rand_score(labels1, labels2)

def embeddings_to_partition(embeddings: np.ndarray, n_communities: int) -> Dict[int, int]:
    """Convert node embeddings to community partition using k-means"""
    if n_communities > len(embeddings):
        n_communities = max(2, len(embeddings) // 5)
    
    kmeans = KMeans(n_clusters=n_communities, random_state=42, n_init=10)
    labels = kmeans.fit_predict(embeddings)
    
    return {i: int(labels[i]) for i in range(len(labels))}

# ============================================================
# 4. EMBEDDING METHODS
# ============================================================

def deepwalk_embedding(G: nx.Graph, dimensions: int = 64, walk_length: int = 30, 
                       num_walks: int = 200, window_size: int = 10) -> np.ndarray:
    """
    DeepWalk embedding using random walks + Word2Vec
    """
    try:
        from gensim.models import Word2Vec
        
        nodes = list(G.nodes())
        walks = []
        
        for _ in range(num_walks):
            np.random.shuffle(nodes)
            for node in nodes:
                walk = [node]
                current = node
                for _ in range(walk_length - 1):
                    neighbors = list(G.neighbors(current))
                    if not neighbors:
                        break
                    current = np.random.choice(neighbors)
                    walk.append(current)
                walks.append([str(w) for w in walk])
        
        model = Word2Vec(sentences=walks, vector_size=dimensions, 
                        window=window_size, min_count=0, sg=1, workers=1, epochs=10)
        
        embeddings = np.array([model.wv[str(node)] for node in nodes])
        return embeddings
    
    except ImportError:
        print("  gensim not installed. Using random embeddings.")
        return np.random.randn(G.number_of_nodes(), dimensions)

def node2vec_embedding(G: nx.Graph, dimensions: int = 64, walk_length: int = 30,
                       num_walks: int = 200, p: float = 1.0, q: float = 1.0,
                       window_size: int = 10) -> np.ndarray:
    """
    Node2Vec embedding using biased random walks + Word2Vec
    """
    try:
        from gensim.models import Word2Vec
        
        def node2vec_walk(start_node, walk_length, p, q):
            walk = [start_node]
            current = start_node
            previous = None
            
            for _ in range(walk_length - 1):
                neighbors = list(G.neighbors(current))
                if not neighbors:
                    break
                
                if previous is None:
                    next_node = np.random.choice(neighbors)
                else:
                    probabilities = []
                    for neighbor in neighbors:
                        if neighbor == previous:
                            prob = 1.0 / p
                        elif G.has_edge(neighbor, previous):
                            prob = 1.0
                        else:
                            prob = 1.0 / q
                        probabilities.append(prob)
                    
                    probabilities = np.array(probabilities)
                    probabilities /= probabilities.sum()
                    next_node = np.random.choice(neighbors, p=probabilities)
                
                walk.append(next_node)
                previous = current
                current = next_node
            
            return walk
        
        nodes = list(G.nodes())
        walks = []
        
        for _ in range(num_walks):
            np.random.shuffle(nodes)
            for node in nodes:
                walk = node2vec_walk(node, walk_length, p, q)
                walks.append([str(w) for w in walk])
        
        model = Word2Vec(sentences=walks, vector_size=dimensions,
                        window=window_size, min_count=0, sg=1, workers=1, epochs=10)
        
        embeddings = np.array([model.wv[str(node)] for node in nodes])
        return embeddings
    
    except ImportError:
        print("  gensim not installed. Using random embeddings.")
        return np.random.randn(G.number_of_nodes(), dimensions)

def graphsage_embedding(G: nx.Graph, dimensions: int = 64) -> np.ndarray:
    """
    GraphSAGE-inspired embedding using neighborhood aggregation
    """
    try:
        from gensim.models import Word2Vec
        
        nodes = list(G.nodes())
        walks = []
        
        # Generate random walks with neighbor sampling
        for _ in range(100):
            np.random.shuffle(nodes)
            for node in nodes:
                walk = [node]
                current = node
                for _ in range(20 - 1):
                    neighbors = list(G.neighbors(current))
                    if not neighbors:
                        break
                    # Sample neighbors (like GraphSAGE sampling)
                    sampled_neighbors = np.random.choice(neighbors, size=min(5, len(neighbors)), replace=False)
                    current = np.random.choice(sampled_neighbors)
                    walk.append(current)
                walks.append([str(w) for w in walk])
        
        model = Word2Vec(sentences=walks, vector_size=dimensions,
                        window=5, min_count=0, sg=1, workers=1, epochs=10)
        
        embeddings = np.array([model.wv[str(node)] for node in nodes])
        return embeddings
    
    except ImportError:
        print("  gensim not installed. Using random embeddings.")
        return np.random.randn(G.number_of_nodes(), dimensions)

# ============================================================
# 5. EMBEDDING METHODS WITH K-MEANS
# ============================================================

def deepwalk_community_detection(G: nx.Graph) -> Dict[int, int]:
    """DeepWalk + k-means for community detection"""
    print("  Running DeepWalk + k-means...")
    n = G.number_of_nodes()
    embeddings = deepwalk_embedding(G)
    n_communities = max(2, int(np.sqrt(n)))
    partition = embeddings_to_partition(embeddings, n_communities)
    return partition

def node2vec_community_detection(G: nx.Graph) -> Dict[int, int]:
    """Node2Vec + k-means for community detection"""
    print("  Running Node2Vec + k-means...")
    n = G.number_of_nodes()
    embeddings = node2vec_embedding(G)
    n_communities = max(2, int(np.sqrt(n)))
    partition = embeddings_to_partition(embeddings, n_communities)
    return partition

def graphsage_community_detection(G: nx.Graph) -> Dict[int, int]:
    """GraphSAGE + k-means for community detection"""
    print("  Running GraphSAGE + k-means...")
    n = G.number_of_nodes()
    embeddings = graphsage_embedding(G)
    n_communities = max(2, int(np.sqrt(n)))
    partition = embeddings_to_partition(embeddings, n_communities)
    return partition

# ============================================================
# 6. MAIN TESTING FUNCTION
# ============================================================

def test_embedding_methods(dataset_name: str, G: nx.Graph, ground_truth: Dict[int, int]):
    """Test all embedding methods on a single dataset"""
    
    methods = {
        "DeepWalk": deepwalk_community_detection,
        "Node2Vec": node2vec_community_detection,
        "GraphSAGE": graphsage_community_detection,
    }
    
    results = []
    
    print(f"\n{'='*80}")
    print(f"Testing Embedding Methods on: {dataset_name}")
    print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")
    if ground_truth:
        print(f"Ground truth communities: {len(set(ground_truth.values()))}")
    print(f"{'='*80}\n")
    
    for name, method in methods.items():
        try:
            start_time = time.time()
            partition = method(G)
            end_time = time.time()
            runtime = end_time - start_time
            
            n_communities = len(set(partition.values()))
            modularity = calculate_modularity(G, partition)
            
            has_gt = ground_truth and len(ground_truth) > 0
            if has_gt:
                nmi = calculate_nmi(partition, ground_truth)
                ari = calculate_ari(partition, ground_truth)
            else:
                nmi = -1
                ari = -1
            
            if has_gt:
                print(f"✓ {name:25s} | Comm: {n_communities:3d} | Mod: {modularity:.4f} | NMI: {nmi:.4f} | ARI: {ari:.4f} | Time: {runtime:.2f}s")
            else:
                print(f"✓ {name:25s} | Comm: {n_communities:3d} | Mod: {modularity:.4f} | NMI: N/A | ARI: N/A | Time: {runtime:.2f}s")
            
            results.append({
                "Algorithm": name,
                "Communities": n_communities,
                "Modularity": round(modularity, 4),
                "NMI": round(nmi, 4) if has_gt else "N/A",
                "ARI": round(ari, 4) if has_gt else "N/A",
                "Runtime (s)": round(runtime, 2)
            })
            
        except Exception as e:
            print(f"✗ {name:25s} | FAILED: {str(e)[:60]}")
            results.append({
                "Algorithm": name,
                "Communities": "ERROR",
                "Modularity": "ERROR",
                "NMI": "ERROR",
                "ARI": "ERROR",
                "Runtime (s)": "ERROR"
            })
    
    return pd.DataFrame(results)

# ============================================================
# 7. MAIN EXECUTION
# ============================================================

def main():
    print("\n" + "="*80)
    print("EMBEDDING METHODS COMMUNITY DETECTION COMPARISON")
    print("Methods: DeepWalk, Node2Vec, GraphSAGE (Embeddings + k-means)")
    print("Metrics: Modularity, NMI, ARI")
    print("Master's Research Project - MOUDDEN Hamza")
    print("="*80)
    
    datasets = {
        "Karate Club": ("karate.gml", load_karate_club),
        "Dolphins": ("Dolphins.gml", load_dolphins),
        "Football": ("Football.gml", load_football),
        "Polbooks": ("Polbooks.gml", load_polbooks),
        "Primary School": ("PS.gml", load_primary_school),
        "VS13": ("VS13.gml", load_vs13),
        "VS15": ("VS15.gml", load_vs15),
    }
    
    all_results = {}
    
    for name, (filename, loader) in datasets.items():
        filepath = get_data_path(filename)
        if os.path.exists(filepath):
            print(f"\nLoading {name} from {filename}...")
            try:
                G, gt = loader()
                if G.number_of_nodes() == 0:
                    print(f"  Error: Graph has no nodes")
                    continue
                df = test_embedding_methods(name, G, gt)
                all_results[name] = df
                safe_name = name.replace(' ', '_').replace('(', '').replace(')', '')
                df.to_csv(f"embedding_results_{safe_name}.csv", index=False)
                print(f"\nResults saved to: embedding_results_{safe_name}.csv")
            except Exception as e:
                print(f"  Error loading {name}: {e}")
        else:
            print(f"\n⚠ File not found: {filepath}")
    
    # Create summary
    if all_results:
        print("\n" + "="*80)
        print("SUMMARY: Average Performance Across All Datasets")
        print("="*80)
        
        summary = []
        methods_order = ["DeepWalk", "Node2Vec", "GraphSAGE"]
        
        for algo in methods_order:
            mod_values = []
            nmi_values = []
            ari_values = []
            time_values = []
            
            for df in all_results.values():
                row = df[df["Algorithm"] == algo]
                if len(row) > 0 and row["Modularity"].iloc[0] != "ERROR":
                    try:
                        mod_values.append(float(row["Modularity"].iloc[0]))
                        if row["NMI"].iloc[0] != "N/A":
                            nmi_values.append(float(row["NMI"].iloc[0]))
                        if row["ARI"].iloc[0] != "N/A":
                            ari_values.append(float(row["ARI"].iloc[0]))
                        time_values.append(float(row["Runtime (s)"].iloc[0]))
                    except:
                        pass
            
            summary.append({
                "Algorithm": algo,
                "Avg Modularity": f"{np.mean(mod_values):.4f}" if mod_values else "N/A",
                "Avg NMI": f"{np.mean(nmi_values):.4f}" if nmi_values else "N/A",
                "Avg ARI": f"{np.mean(ari_values):.4f}" if ari_values else "N/A",
                "Avg Time (s)": f"{np.mean(time_values):.2f}" if time_values else "N/A"
            })
        
        summary_df = pd.DataFrame(summary)
        print(summary_df.to_string(index=False))
        summary_df.to_csv("embedding_results_summary.csv", index=False)
        print("\nSummary saved to: embedding_results_summary.csv")
    
    print("\n" + "="*80)
    print("Embedding methods execution completed.")
    print("="*80)

if __name__ == "__main__":
    main()


EMBEDDING METHODS COMMUNITY DETECTION COMPARISON
Methods: DeepWalk, Node2Vec, GraphSAGE (Embeddings + k-means)
Metrics: Modularity, NMI, ARI
Master's Research Project - MOUDDEN Hamza

Loading Karate Club from karate.gml...

Testing Embedding Methods on: Karate Club
Nodes: 34, Edges: 78
Ground truth communities: 2

  Running DeepWalk + k-means...
  gensim not installed. Using random embeddings.
✓ DeepWalk                  | Comm:   5 | Mod: 0.1137 | NMI: 0.2790 | ARI: 0.2154 | Time: 3.19s
  Running Node2Vec + k-means...
  gensim not installed. Using random embeddings.
✓ Node2Vec                  | Comm:   5 | Mod: -0.0408 | NMI: 0.0713 | ARI: 0.0002 | Time: 0.03s
  Running GraphSAGE + k-means...
  gensim not installed. Using random embeddings.
✓ GraphSAGE                 | Comm:   5 | Mod: -0.0654 | NMI: 0.0886 | ARI: 0.0044 | Time: 0.05s

Results saved to: embedding_results_Karate_Club.csv

Loading Dolphins from Dolphins.gml...

Testing Embedding Methods on: Dolphins
Nodes: 62, Edges: